# Chapter 9: Transformers

Knowledge of temporal information.

Sinusoidal Position Embedding implementation

In [ ]:
import math
import numpy as np
class PositionalEncodingSimple:
    """Positional encoding for our transformer
    written in numpy.
    :param d_model: hidden dimension of the encoder
    :param max_len: maximum length of our positional
    encoder. The encoder can not encode sequence
    length longer than max_len.
    """
    def __init__(self, d_model: int, max_len: int = 100):
        position = np.expand_dims(np.arange(max_len), axis=1)
        div_term = np.exp(np.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = np.zeros((max_len, d_model))
        self.pe[:, 0::2] = np.sin(position * div_term)
        self.pe[:, 1::2] = np.cos(position * div_term)

    def __call__(self, x: np.ndarray) -> np.ndarray:
        """
        :param x: input to be encoded
        with shape
        `(batch_size, sequence_length, embedding_dim)`
        """
        return self.pe[: x.shape[1]]

The previous implementation is to help us build up the intuitions. To be able to use positional encoding in a transformer model, we need to implement it as a torch.nn.Module. The following is one
implementation of the positional encoding using PyTorch which can be plugged into a transformer
model. We also included dropout as in the original transformer implementation.

In [ ]:
from torch import nn
import torch
class PositionalEncoding(nn.Module):
    """Positional encoding to be added to
    input embedding.
    :param d_model: hidden dimension of the encoder
    :param dropout: rate of dropout
    :param max_len: maximum length of our positional
    encoder. The encoder can not encode sequence
    length longer than max_len.
    """
    def __init__(
    self, d_model: int,
    dropout: float = 0.1,
    max_len: int = 5000,
    ):
        super().__init__()
        self.max_len = max_len
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        :param x: input embedded time series,
        shape `[batch_size, seq_len, embedding_dim]`
        """
        history_length = x.size(1)
        x = x + self.pe[: history_length]
        return self.dropout(x)

### Implementation of transformers

1. First, we define a dataclass that holds all the hyperparameters of the model. A dataclass is
very useful when passing the hyperparameters to our model.

In [ ]:
import dataclasses

@dataclasses.dataclass
class TSTransformerParams:
    """A dataclass that contains all
    the parameters for the transformer model.
    """
    d_model: int = 512
    nhead: int = 8
    num_encoder_layers: int = 6
    dropout: int = 0.1

2. We implement a simple transformer for time series, which consists of embedding, encoder,
and decoder. We choose the simplest approaches for pedagogical purposes. In the following
code block, we use a linear layer as embedding, which expands each time series value into a
long vector in the embedding space. We skip the details of the attention implementation in
an encoder and take nn.TransformerEncoder from PyTorch directly. The decoder is also a
linear layer.

In [ ]:
class TSTransformer(nn.Module):
    """Transformer for univariate time series forecasting.
    :param history_length: the length of the input history.
    :param horizon: the number of steps to be forecasted.
    :param transformer_params: all the parameters.
    """
    def __init__(
    self,
    history_length: int,
    horizon: int,
    transformer_params: TSTransformerParams,
    ):
        super().__init__()
        self.transformer_params = transformer_params
        self.history_length = history_length
        self.horizon = horizon
        self.embedding = nn.Linear(
        1, self.transformer_params.d_model
        )
        encoder_layer = nn.TransformerEncoderLayer(
             d_model=self.transformer_params.d_model,
        nhead=self.transformer_params.nhead,
        batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(
        encoder_layer,
        num_layers=self.transformer_params.num_encoder_layers
        )
        self.reverse_embedding = nn.Linear(
        self.transformer_params.d_model, 1
        )
        self.decoder = nn.Linear(
        self.history_length, self.horizon
        )

    @property
    def transformer_config(self) -> dict:
        """all the param in dict format
        """
        return dataclasses.asdict(
        self.transformer_params
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        :param x: input historical time series,
        shape `[batch_size, seq_len, n_var]`
        """
        x = self.embedding(x)
        encoder_state = self.encoder(x)
        decoder_in = self.reverse_embedding(
        encoder_state
        ).squeeze(-1)
        return self.decoder(decoder_in)

3. Similar to our RNN implementations and experiments, we also create a class to hold all the
training, validation, and prediction functionalities

In [ ]:
import lightning as L
class TransformerForecaster(L.LightningModule):
    """Transformer forecasting training, validation,
    and prediction all collected in one class.
    :param transformer: pre-defined transformer model
    """
    def __init__(self, transformer: nn.Module):
        super().__init__()
        self.transformer = transformer

    def configure_optimizers(self)-> torch.optim.Optimizer:
        optimizer = torch.optim.SGD(self.parameters(), lr=1e-3)
        return optimizer

    def training_step(self, batch: tuple[torch.Tensor], batch_idx: int) -> torch.Tensor:
        x, y = batch
        y = y.squeeze(-1).type(self.dtype)
        y_hat = self.transformer(x)
        loss = nn.functional.mse_loss(y_hat, y)
        self.log_dict(
        {"train_loss": loss},
        prog_bar=True
        )
        return loss

    def validation_step(self, batch: tuple[torch.Tensor], batch_idx: int) -> torch.Tensor:
        x, y = batch
        y = y.squeeze(-1).type(self.dtype)
        y_hat = self.transformer(x)
        loss = nn.functional.mse_loss(y_hat, y)
        self.log_dict(
        {"val_loss": loss},
        prog_bar=True
        )
        return loss

    def predict_step(self, batch: list[torch.Tensor], batch_idx: int)-> tuple[torch.Tensor]:
        x, y = batch
        y = y.squeeze(-1).type(self.dtype)
        y_hat = self.transformer(x)
        return x, y_hat

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor]:
        return x, self.transformer(x)

4. In this example, we set the history input length to be 100 to be able to capture some trends in
the input data and our objective is to forecast one step in the future, i.e., the forecast horizon
is set to 1

In [ ]:
history_length_1_step = 100
horizon_1_step = 1

5. The hyperparameters can be set using the dataclass TSTransformerParams

In [ ]:
ts_transformer_params_1_step = TSTransformerParams(
 d_model=192, nhead=6, num_encoder_layers=1
)
ts_transformer_params_1_step
TSTransformerParams(
 d_model=192, nhead=6, num_encoder_layers=1, dropout=0.1
)

6. With the hyperparameters defined, we pass them on to our transformer TSTransformer. The
layers are printed out to read about the details inside the encoder layer.

In [ ]:
ts_transformer_1_step = TSTransformer(
 history_length=history_length_1_step,
 horizon=horizon_1_step,
 transformer_params=ts_transformer_params_1_step,
)
ts_transformer_1_step

7. The transformer can be inserted into the forecaster TransformerForecaster and can be trained
exactly the same way as we have shown in the previous chapter

In [ ]:
transformer_forecaster_1_step = TransformerForecaster(
 transformer=ts_transformer_1_step
)

### The M5 competition

In this example, we use the M5 competition dataset. The M5 competition is the fifth M competition
focusing on sales forecasting based on the sales of Walmart (more details about the competition can
be found here https://www.sciencedirect.com/science/article/pii/S0169207021001187) (Makridakis, Spiliotis, and Assimakopoulos 2022). We simplify the dataset by selecting only California stores
and aggregate the sales to the state level, thus we only forecast one single time series: the daily sales
of all Walmart stores in California. The following is a small sample of the data.

In the following graphs, the upper panel shows the full time series of sales in all California stores
in Walmart. The middle panel shows the details of the year 2015. We observe the cyclic features of
the time series. The bottom panel plots the autocorrelation of the time series. As expected, there is
a 7-day cycle in sales.

Our objective is to build a transformer model that forecasts the sales of the next 3 days. To do this, we
will follow these steps:

1. To be able to justify the performance of our model, we split the data at 2016-01-01 using the code:

In [ ]:
import pandas as pd
df = pd.read_csv("data/m5_california_sales.csv")

train_test_split_date = "2016-01-01"
df_train = df[df.ds<=train_test_split_date]
df_test = df[df.ds>train_test_split_date]
horizon = 3

2. Before we start building the transformer forecaster, we construct a baseline forecaster. Here
we choose an ARIMA model as we have clear cyclic behavior in the data. We stick to the
Nixtlaverse and use their statsforecast package.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
sf = StatsForecast(
 models = [AutoARIMA(
 seasonal=False
 )],
 freq = 'd'
)

3. We also create a function to help us evaluate the tests. We perform the test with a fixed horizon,
but the input time series is sliding into the future with a step size of 1 day.

In [ ]:
import pandas as pd
import logging
logger = logging.getLogger(__name__)

def forecast_test(forecaster_pred: callable, df_train: pd.DataFrame, df_test: pd.DataFrame) -> pd.DataFrame:
    """Generate forecasts for tests
    We gradually increase the historical information
    by appending part of the test data and
    predict based on the new historical information.
    :param forecaster_pred: forecasting function/method
    :param df_train: train dataframe
    :param df_test: test dataframe
    """
    dfs_pred = []
    for i in range(len(df_test)):
        logger.debug(f"Prediction Step: {i}")
        df_pred_input_i = pd.concat([df_train, df_test[:i]]).reset_index(drop=True)
        df_pred_output_i = forecaster_pred(df_pred_input_i)
        df_pred_output_i["step"] = i
        dfs_pred.append(df_pred_output_i)

    df_y_hat = pd.concat(dfs_pred)
    return df_y_hat

4. We obtain all the baseline model forecasts by calling the forecast method of a StatsForecast
model:

In [ ]:
df_y_hat_arima = forecast_test(
 lambda x: sf.forecast(df=x, h=horizon),
 df_train=df_train,
 df_test=df_test
)

The baseline forecasts from ARIMA are not bad at all. It is expected because sales data is indeed seasonal and ARIMA is good at capturing seasonalities.

With this baseline ready, we will start building our transformer forecaster. It is extremely easy to
create a transformer forecaster in Nixtla’s neuralforecast. In fact, we are creating two transformer
models, VanillaTransformer and iTransformer. iTransformer is just like a vanilla transformer, but
instead of embedding each time step, it transposes the data and embeds each time series independently
(iTransformer: Inverted Transformers Are Effective for Time Series Forecasting by Yong Liu et al., available
at https://arxiv.org/abs/2310.06625) (Liu et al. 2023).

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import VanillaTransformer, iTransformer
models = [
 VanillaTransformer(
 hidden_size=128,
 n_head=4,
 learning_rate=0.0001,
 scaler_type="robust",
 max_steps=500,
 batch_size=32,
 windows_batch_size=512,
 random_seed=16,
 input_size=30,
 step_size=3,
 h=horizon,
 ),
 iTransformer(
 input_size=30,
 h=horizon,
 max_steps=50,
 n_series=1,
 )
]
nf = NeuralForecast(models=models, freq="d")

Model training is a one-liner: calling the fit method. Behind the scenes, what happens is that the fit
method calls the Trainer from lightning package, which we have been using to build the forecasters
in the previous section as well as the previous chapter. Nixtla’s neuralforecast is handy because we
don’t need to manually create the trainer and perform the training.

In [ ]:
nf.fit(df=df_train)

We generate the forecasts for the whole test range similar to what we have calculated for the baseline
model.

In [ ]:
df_y_hat_transformer = forecast_test(
 nf.predict, df_train=df_train, df_test=df_test
)

Transformers are performing better than our baseline, judging by the distribution of the metrics on
the full test dataset.

In Figure 9.10, we observe that our transformers are better than the baseline. What is even more
interesting is that we only trained our transformer model once and only using the data before 2016,
while we have retrained our ARIMA model for each evaluation. In principle, the ARIMA baseline has
utilized more data than our transformer models. But it still falls behind the transformers. The reason
that transformers are better in this example is that transformers forecast better for more distant
future steps:

As you can see, ARIMA is visually much worse than the transformers.